In [1]:
import numpy as np
import cv2
import time
import matplotlib.pyplot as plt
from PIL import Image
import os
from paddle.vision.transforms import functional as F
import json
import base64


In [2]:
def get_color_map_list(num_classes):
    """
    Returns the color map for visualizing the segmentation mask,
    which can support arbitrary number of classes.
    Args:
        num_classes (int): Number of classes.
    Returns:
        (list). The color map.
    """

    num_classes += 1
    color_map = num_classes * [0, 0, 0]
    for i in range(0, num_classes):
        j = 0
        lab = i
        while lab:
            color_map[i * 3] |= (((lab >> 0) & 1) << (7 - j))
            color_map[i * 3 + 1] |= (((lab >> 1) & 1) << (7 - j))
            color_map[i * 3 + 2] |= (((lab >> 2) & 1) << (7 - j))
            j += 1
            lab >>= 3
    color_map = color_map[3:]
    return color_map

In [3]:
color_map = get_color_map_list(256)

In [4]:
src_img_path = "C:\\Users\\hp\\Desktop\\4_rect_1\\1024\\leftImg8bit"
# src_img_path = "C:\\Users\\hp\\Desktop\\4_rect_1\\bg"
src_mask_path = "C:\\Users\\hp\\Desktop\\4_rect_1\\1024\\gtFine"

# save_img_path = "C:\\Users\\hp\\Desktop\\4_aug\\leftImg8bit"
save_img_path = "C:\\Users\\hp\\Desktop\\4_rect_1\\512\\leftImg8bit"
save_mask_path = "C:\\Users\\hp\\Desktop\\4_rect_1\\512\\gtFine"

In [5]:
imgs_name = []
for _item in os.listdir(src_img_path):
    if _item.split('.')[-1] in ['jpg', 'png', 'bmp', 'jpeg']:
        imgs_name.append(_item)

In [10]:
demo_img = np.array(Image.open(os.path.join(src_img_path, imgs_name[0])))

In [6]:
# new_shape = (1024,1024)
new_shape = (512,512)

for _imgName in imgs_name:
    _maskName = _imgName.split('.')[0] + '.png'
    
    img = Image.open(os.path.join(src_img_path, _imgName))#加载图片
    # img_bg = Image.open(os.path.join(src_bg_path, _imgName))#加载图片
    img_mask = Image.open(os.path.join(src_mask_path, _maskName))
    
    img = np.array(img)
    img = cv2.resize(img, new_shape)
    converted_img = Image.fromarray(img)
    
    # converted_img = img.resize([size,size], resample=Image.BICUBIC)
    _new_img_name = _imgName
    converted_img.save(os.path.join(save_img_path, _new_img_name))
    
    #保存标签
    img_mask = np.array(img_mask)
    img_mask = cv2.resize(img_mask, new_shape)
    lbl_pil = Image.fromarray(img_mask)
    # lbl_pil = img_mask.resize([size,size], resample=Image.NEAREST)
    lbl_pil.putpalette(color_map)
    _new_mask_name = _maskName
    lbl_pil.save(os.path.join(save_mask_path, _new_mask_name))

In [7]:

def imageToStr(image):
    with open(image,'rb') as f:
        image_byte=base64.b64encode(f.read())
        # print(type(image_byte))
    image_str=image_byte.decode('ascii') #byte类型转换为str
    # print(type(image_str))
    return image_str

In [8]:
json_names = []
for _item in os.listdir(src_img_path):
    if _item.split('.')[-1] in ['json']:
        json_names.append(_item)

In [11]:
# size = 1024.0
size = 512.0
rows_y = demo_img.shape[0]
cols_x = demo_img.shape[1]
transition1 = np.array([[cols_x/2.0, rows_y/2.0]], dtype=np.float32)
transition2 = np.array([[size/2.0, size/2.0]], dtype=np.float32)
M = np.array([[size/cols_x, 0.0], [0.0, size/rows_y]], dtype=np.float32)

for jsonName in json_names:
    with open(os.path.join(src_img_path, jsonName), 'r+') as f:
        data = json.load(f)
        data2 = data.copy()
    
    #转换点
    for i in range(0, len(data['shapes'])):
        one_cout = np.array(data['shapes'][i]['points'])
        
        one_cout_numpy = np.array(one_cout, dtype=np.float32)
        one_cout_numpy = one_cout_numpy - transition1 # 平移到原点
        one_cout_numpy_trans = one_cout_numpy @ M
        one_cout_numpy_trans = one_cout_numpy_trans + transition2
        one_cout_trans = one_cout_numpy_trans.astype(np.int32)
        one_cout_trans = one_cout_trans.tolist()
        
        data2['shapes'][i]['points'] = one_cout_trans
        
    # 替换图片, 要求文件夹必须已经有图片
    _imgName = jsonName.split('.')[0] + '.jpg'
    data2['imageData'] = imageToStr(os.path.join(save_img_path, _imgName))
    
    data2['imageHeight'] = int(size)
    data2['imageWidth'] = int(size)
    
    with open(os.path.join(save_img_path, jsonName), 'w') as f:
        json.dump(data2, f)
    

In [ ]:
# json resize
with open(os.path.join(src_img_path, '1.json'), 'r+') as f:
    data = json.load(f)

In [ ]:
print(type(data['shapes'][0]['points']))
print(np.array(data['shapes'][0]['points']).shape)
print(len(data['shapes']))
print(data['imagePath'])
print(data.keys())

In [ ]:
print(data['imageData'])

In [ ]:
data2 = data.copy()

In [ ]:
one_cout = np.array(data2['shapes'][0]['points'])

In [ ]:
size = 1024.0
rows_y = demo_img.shape[0]
cols_x = demo_img.shape[1]
transition1 = np.array([[cols_x/2.0, rows_y/2.0]], dtype=np.float32)
transition2 = np.array([[size/2.0, size/2.0]], dtype=np.float32)
M = np.array([[size/cols_x, 0.0], [0.0, size/rows_y]], dtype=np.float32)

one_cout = np.array(one_cout, dtype=np.float32)
one_cout_numpy = np.array(one_cout, dtype=np.float32)
one_cout_numpy = one_cout_numpy - transition1 # 平移到原点
one_cout_numpy_trans = one_cout_numpy @ M
one_cout_numpy_trans = one_cout_numpy_trans + transition2
one_cout_trans = one_cout_numpy_trans.astype(np.int32)
one_cout_trans = one_cout_trans.tolist()

In [ ]:
data2['shapes'][0]['points'] = one_cout_trans

In [ ]:
print(data2['shapes'][0]['points'])


In [ ]:
_demo = cv2.imread(os.path.join(save_img_path, '1.jpg'))
print(_demo.shape)
cv2.drawContours(_demo, [np.array(data2['shapes'][0]['points'])], 0, (0,255,0), thickness=2) #画原轮廓，绿色
plt.figure(figsize=(9,9))
plt.imshow(cv2.cvtColor(_demo, cv2.COLOR_BGR2RGB))

In [ ]:
with open(os.path.join(save_img_path, '1.json'), 'w') as f:
    json.dump(data2, f)